### GROUPWORK ASSIGNMENT

Titanic Survival Prediction

In [ ]:
# importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
data = pd.read_csv("tested.csv")
data.head()

In [ ]:
# this remopves spaces from the column headings
data.columns = data.columns.str.strip()
data.head()

In [ ]:
# fills missing values (NaNs) in a Pandas DataFrame with the median of each numeric column.
data = data.fillna(data.median(numeric_only=True))
data.info()

In [ ]:
# converting column sex in into numerical male:1, female:0 
# the first line in code enhances that the sex.dtype changes from object to int
if 'Sex' in data.columns and data['Sex'].dtype == 'object':
    data['Sex'] = data['Sex'].map({'male': 1, 'female': 0})
data.info()

In [ ]:
# selects all the columns in data that contain numbers only and stores them in a new DataFrame called numeric_data.
numeric_data = data.select_dtypes(include=['number'])

In [ ]:
# features and targets -- survived is the target the other columns are the features which ill use to predict the outcome in survived.
X = numeric_data.drop(columns=['Survived'])
y = numeric_data['Survived']

In [ ]:
# Train	- Where your model learns it sees this data and adjusts its weights.
# Validation (Val)-	Used to tune the model eg picking the best number of trees, learning rate, or regularization. 
# The model does not learn from validation, it only uses it for feedback.
# Test - Used at the very end to measure true performance — it's data the model has never seen and never used in any way during training.
# x_test, y_test: This is your final test set (15% of all data)
# x_temp, y_temp: This is the temporary dataset holding the remaining 85% of data — which you'll later split into train and validation
# That's why it's called temp(temporary) it’s not a final split, just an intermediate result.
# x_train, y_train → 70% of the original data (used to train the model)
# x_val, y_val → 15% of the original data (used to tune the model)
# Why 0.1765? Because you're splitting 85% into 70/15, so:15/85 = 0.1765
x_temp, x_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

In [ ]:
# model = LogisticRegression(max_iter=1000)-- Creates a logistic regression model.
# max_iter=1000 increases the number of iterations the solver will try to converge to find a solution.
# model.fit(x_train, y_train) -Trains the model using the training features x_train and labels y_train.
# The model learns the relationship between the features and the target (like which variables influence survival).
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

In [ ]:
y_val_pred = model.predict(x_val)
y_val_probs = model.predict_proba(x_val)[:, 1] #pick 2nd column
#''' It gives probabilities for each class, e.g. 
# [[0.83, 0.17],
# [0.12, 0.88],
# [0.45, 0.55]]
# where:
# First column = probability of class 0 (did not survive)
# Second column = probability of class 1 (survived)
# '''
print (y_val_pred)
print (y_val_probs)

In [ ]:
# Accuracy-	How often the model is correct overall	"Out of all predictions, how many were right?"
# Precision- When the model predicts survived (1), how often is it actually right? "Of all the people it predicted would survive, how many really did?"
# Recall- Out of all who actually survived, how many did the model catch?	"Did the model find all the survivors?"
# F1 Score-	A balanced average of precision and recall	"One score that balances precision and recall (especially useful if data is imbalanced)"
metrics = {
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall": recall_score(y_val, y_val_pred),
    "F1 Score": f1_score(y_val, y_val_pred)
}

print (metrics)

In [ ]:
# print("\n MODEL PERFORMANCE ON VALIDATION SET ") -Just a title to separate output clearly.
# \n adds a blank line before the title for readability.
# for k, v in metrics.items()-Loops through your metrics dictionary
# print(f"{k}: {v:.3f}")- Prints the metric name (k) and its value (v) rounded to 3 decimal places.
print("\n MODEL PERFORMANCE ON VALIDATION SET ")
for k, v in metrics.items():
    print(f"{k}: {v:.3f}")

In [ ]:
y_test_pred = model.predict(x_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"\nTest Accuracy (Unseen Data): {test_accuracy:.3f}")

In [ ]:
# heatmap for correlating each column by column
plt.figure(figsize=(8,6)) #Sets the figure size to make the plot readable (8 inches wide, 6 inches tall).
corr = numeric_data.corr() # Calculates the correlation matrix for all numeric columns.This gives values between -1 and 1: -1 -perfect -ve correlation ,0 no correlation, +1 -perfect positive correlation
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f') # annot shows correlation values inside each cell, cmap is for the colors , fmt for decimal places
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
feature_importance = pd.Series(model.coef_[0], index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8,6))
feature_importance.plot(kind='barh')
plt.title("Feature Importance (Logistic Regression Coefficients)")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.show()

In [ ]:
print("\nUsing validation dataset for demonstration of likely survivors...")
predictions = pd.DataFrame({
    'Predicted_Probability': y_val_probs,
    'Actual_Survived': y_val.reset_index(drop=True) #since we randomly selected data, this ensures pandas does not try to arrange by indexes which might cause mismatch issues
})
predictions['Prediction_Label'] = predictions['Predicted_Probability'].apply(
    lambda p: 'Likely to Survive' if p >= 0.5 else 'Likely Not to Survive'
)
print (predictions)

In [ ]:
print("\n SAMPLE PREDICTIONS ")
print(predictions.head(15)

In [ ]:
print("\nSURVIVAL SUMMARY (For our samples)")
print(predictions['Prediction_Label'].value_counts())

In [ ]:
full_pred = model.predict(X)
full_probs = model.predict_proba(X)[:, 1]
print (full_pred)
print (full_probs)

In [ ]:
data['Predicted_Survival'] = full_pred
data['Survival_Probability'] = full_probs
data.head()

In [ ]:
if 'Predicted_Survival' in data.columns and data['Predicted_Survival'].dtype == 'int':
    data['Predicted_Survival'] = data['Predicted_Survival'].map({1: 'Survived', 0: 'didnotsurvive'})
data.head()